# M3 · Products Cleaning (Excel: Power Query, formulas, pivot)

**Goal:** clean the product catalog (500 rows) **inside Excel**, the way a client who lives in Excel can open, audit and refresh it.

This notebook only **drives Excel** (through its automation interface) to build `clean_data/products_clean.xlsx`, then reads the result back to validate it. All cleaning logic lives in the workbook itself:

| Sheet | What it does |
|---|---|
| `notes` | how the workbook works and how to refresh it |
| `raw_products` | the untouched raw CSV as table `tbl_raw` (all text) |
| `category_map` | lookup table `tbl_category_map`: every category variant → one of 8 standard categories |
| `products_clean` | Power Query output `tbl_products` + formula columns (XLOOKUP, LET, IFS, SUMPRODUCT, TEXTJOIN); helper columns grouped |
| `flagged` | dynamic `FILTER` (with `CHOOSE` to pick the main columns) of every product with a problem |
| `summary` | pivot tables: products and average price per category, price status counts |

The Power Query code is kept as a readable file next to this notebook: `products_clean.m`.

**Requirement:** Microsoft Excel 2021 / Microsoft 365 on Windows (for Power Query automation, XLOOKUP, IFS, FILTER).

## 1. Setup

In [1]:
import shutil
import tempfile
import time
from pathlib import Path

import pandas as pd
import pythoncom
import win32com.client as win32

RAW_DIR = Path("../../01_data_and_task/raw_data").resolve()
CLEAN_DIR = Path("../../03_clean_data_and_report/clean_data").resolve()
OUT_DIR = Path("output").resolve()  # detailed working outputs of this milestone
OUT_DIR.mkdir(exist_ok=True)
ISSUE_LOG = Path("../01_profiling-pandas/output/issue_log.xlsx")
OUT_PATH = CLEAN_DIR / "products_clean.xlsx"
M_CODE = Path("products_clean.m").read_text(encoding="utf-8")

raw = pd.read_csv(RAW_DIR / "product_catalog_dirty_30pct.csv", dtype=str, keep_default_na=False, encoding="utf-8")
issue_log = pd.read_excel(ISSUE_LOG, sheet_name="issue_log")
m3_issues = issue_log[issue_log["milestone"] == "M3"]
print(f"raw products: {len(raw):,} rows")
m3_issues[["issue_id", "column", "issue", "rows_affected"]]

raw products: 500 rows


,issue_id,column,issue,rows_affected
20,ISS-021,category,non-standard category value (38 variants: case...,108
21,ISS-022,product_name,leading/trailing spaces,37
22,ISS-023,price,leading/trailing spaces,35
23,ISS-024,category,blank value,30
24,ISS-025,product_name,noise characters (digits or symbols) in name,30
25,ISS-026,price,non-numeric price (thousand separator) (all ro...,30
26,ISS-027,price,placeholder text instead of a value,26
27,ISS-028,price,negative price (all rows hold the same value '...,26
28,ISS-029,product_name,missing space between words (e.g. 'UsuallyHear'),23
29,ISS-030,price,zero price (all rows hold the same value '0'),19


## 2. Category map

Power Query reduces every raw category to a `category_key` (lower case, `3` → `e`, trailing `_`/`-` removed). The keys that remain are either a standard category or a truncated one (`clo`, `kit`, `ele`...).

The map below lists **every key found in the data** with its standard category and the rule used. A truncated key is mapped only when exactly one standard category starts with it; otherwise it would stay unmapped. The map is a table in the workbook, so a reviewer can see and change it.

In [2]:
STANDARD = ["automotive", "beauty", "clothing", "electronics", "home", "kitchen", "sports", "toys"]

# same steps as the Power Query category_key column
raw_cat = raw["category"].str.strip()
keys = raw_cat.str.lower().str.replace("3", "e").str.rstrip("_-")


def describe(variants):
    rules = []
    if any(v != v.lower() for v in variants): rules.append("case")
    if any("3" in v for v in variants): rules.append("leetspeak 3->e")
    if any(v.endswith(("_", "-")) for v in variants): rules.append("trailing _ or -")
    return rules


rows = []
for key, variants in raw_cat.groupby(keys):
    if key == "":
        continue
    variants = sorted(set(variants))
    matches = [s for s in STANDARD if s.startswith(key)]
    if key in STANDARD:
        category, rule = key, ["exact"] + describe(variants)
    elif len(matches) == 1:
        category, rule = matches[0], ["truncated"] + describe(variants)
    else:
        category, rule = "", ["no unique match"]
    rows.append({"category_key": key, "category": category, "rule": ", ".join(rule),
                 "raw_variants": " | ".join(variants), "rows": int((keys == key).sum())})

category_map = pd.DataFrame(rows)
assert category_map["category"].ne("").all(), "some keys could not be mapped"
category_map

,category_key,category,rule,raw_variants,rows
0,aut,automotive,truncated,aut,1
1,automotive,automotive,"exact, case, leetspeak 3->e, trailing _ or -",AUTOMOTIVE | automotiv3 | automotive | automot...,51
2,bea,beauty,truncated,bea,4
3,beauty,beauty,"exact, case, leetspeak 3->e, trailing _ or -",BEAUTY | b3auty | beauty | beauty- | beauty_,56
4,clo,clothing,truncated,clo,4
5,clothing,clothing,"exact, case, trailing _ or -",CLOTHING | clothing | clothing- | clothing_,65
6,ele,electronics,truncated,ele,3
7,electronics,electronics,"exact, case, leetspeak 3->e, trailing _ or -",3l3ctronics | ELECTRONICS | electronics | elec...,54
8,hom,home,truncated,hom,5
9,home,home,"exact, case, leetspeak 3->e, trailing _ or -",HOME | hom3 | home | home- | home_,54


## 3. Build the workbook in Excel

Steps performed in Excel:
1. Write the raw CSV (as text, so `' 32309.95'` and `'1,200'` stay exactly as delivered) and create table `tbl_raw`.
2. Write `tbl_category_map`.
3. Add the Power Query `products_clean` (code from `products_clean.m`) and load it to table `tbl_products`.
4. Add formula columns to `tbl_products`.
5. Add the `flagged` FILTER sheet, the pivot tables and the notes sheet.
6. Refresh, calculate and save.

In [3]:
XL_SRC_RANGE, XL_SRC_EXTERNAL, XL_YES = 1, 0, 1
XL_CMD_SQL, XL_DATABASE = 2, 1
XL_ROW_FIELD, XL_COUNT, XL_AVERAGE = 1, -4112, -4106

FORMULAS = {
    # standard category from the map; empty when the raw category was blank
    "category": '=IF([@[category_key]]="","",XLOOKUP([@[category_key]],tbl_category_map[category_key],tbl_category_map[category],"UNMAPPED"))',
    # LET: repeats = how many products carry exactly the same raw price text
    # (real prices are unique; injected placeholders repeat)
    "price_status": '=LET(repeats,SUMPRODUCT(--([price_raw]=[@[price_raw]])),'
                    'IFS([@[price_number]]="","invalid: not a number",[@[price_number]]<0,"invalid: negative",'
                    '[@[price_number]]=0,"invalid: zero",repeats>=5,"invalid: repeated placeholder",TRUE,"ok"))',
    "price": '=IF([@[price_status]]="ok",[@[price_number]],"")',
    "review_reason": '=TEXTJOIN("; ",TRUE,IF([@[category]]="","category missing",""),'
                     'IF([@[category]]="UNMAPPED","category not mapped",""),'
                     'IF([@[price_status]]<>"ok","price "&[@[price_status]],""))',
    # every product problem here is an invalid or missing source value (nothing to decide, nothing to recover)
    "flag_type": '=IF([@[review_reason]]="","",IF(ISNUMBER(SEARCH("not mapped",[@[review_reason]])),'
                 '"needs review","invalid or missing value"))',
}
HELPER_COLUMNS = "C:E"  # category_key, price_raw, price_number: Power Query helpers, grouped and collapsed
FLAGGED_COLUMNS = ["product_id", "product_name", "category", "price_raw", "price_status", "flag_type", "review_reason"]

NOTES = [
    ["Products catalog: cleaned in Excel"],
    [""],
    ["How it works"],
    ["1. raw_products: the raw CSV, untouched, as table tbl_raw (all text)."],
    ["2. Power Query 'products_clean' (Data > Queries & Connections) reads tbl_raw and outputs the first 5 columns of tbl_products:"],
    ["   trims spaces, removes digits/symbols from names, splits joined words, fixes the 'aa' typo, proper case,"],
    ["   builds category_key (lower case, 3->e, trailing _/- removed) and converts price text to a number."],
    ["   The helper columns category_key, price_raw, price_number (C:E) are grouped: click + above column F to show them."],
    ["3. Formula columns in tbl_products:"],
    ["   category       XLOOKUP of category_key in tbl_category_map"],
    ["   price_status   LET + SUMPRODUCT + IFS: not a number / negative / zero / repeated placeholder / ok"],
    ["   price          the clean price, only when price_status is ok"],
    ["   review_reason  TEXTJOIN of every problem found in the row"],
    ["   flag_type      'invalid or missing value' (cannot be recovered) or 'needs review' (a person should decide)"],
    ["4. flagged: FILTER of the main columns of tbl_products where review_reason is not empty."],
    ["5. summary: pivot tables on tbl_products (refresh with Data > Refresh All)."],
    [""],
    ["Rules"],
    ["Values are never guessed. Invalid prices become empty and are listed in 'flagged'. Missing categories stay empty."],
    ["To refresh after changing raw data or the category map: Data > Refresh All."],
]


def write_table(ws, df, name, text=True):
    values = [list(df.columns)] + df.astype(str).values.tolist()
    rng = ws.Range(ws.Cells(1, 1), ws.Cells(len(values), len(df.columns)))
    if text:
        rng.NumberFormat = "@"
    rng.Value = values
    lo = ws.ListObjects.Add(XL_SRC_RANGE, rng, None, XL_YES)
    lo.Name = name
    ws.Columns.AutoFit()
    return lo


pythoncom.CoInitialize()
excel = win32.DispatchEx("Excel.Application")
excel.Visible = False
excel.DisplayAlerts = False
try:
    wb = excel.Workbooks.Add()
    while wb.Worksheets.Count < 6:
        wb.Worksheets.Add(After=wb.Worksheets(wb.Worksheets.Count))
    names = ["notes", "raw_products", "category_map", "products_clean", "flagged", "summary"]
    for i, name in enumerate(names, start=1):
        wb.Worksheets(i).Name = name

    # 1-2. raw data and category map
    write_table(wb.Worksheets("raw_products"), raw, "tbl_raw")
    write_table(wb.Worksheets("category_map"), category_map, "tbl_category_map")

    # 3. Power Query, loaded to a table
    wb.Queries.Add("products_clean", M_CODE)
    ws = wb.Worksheets("products_clean")
    conn = ("OLEDB;Provider=Microsoft.Mashup.OleDb.1;Data Source=$Workbook$;"
            "Location=products_clean;Extended Properties=\"\"")
    lo = ws.ListObjects.Add(XL_SRC_EXTERNAL, conn, None, XL_YES, ws.Range("$A$1"))
    lo.Name = "tbl_products"
    qt = lo.QueryTable
    qt.CommandType = XL_CMD_SQL
    qt.CommandText = "SELECT * FROM [products_clean]"
    qt.RowNumbers = False
    qt.PreserveFormatting = True
    qt.Refresh(False)

    # 4. formula columns (the long [@[column]] form is used: this Excel build rejects the short [@column] form via automation)
    for col, formula in FORMULAS.items():
        new_col = lo.ListColumns.Add()
        new_col.Name = col
        new_col.DataBodyRange.Formula2 = formula
    for col in ["price_number", "price"]:
        lo.ListColumns(col).DataBodyRange.NumberFormat = "#,##0.00"
    ws.Columns.AutoFit()
    ws.Range(HELPER_COLUMNS).Font.Color = 0x808080           # grey: helper columns
    ws.Range(HELPER_COLUMNS).Columns.Group()
    ws.Outline.ShowLevels(0, 1)       # collapse the helper group

    # 5a. flagged sheet: header row + dynamic FILTER of the main columns only
    #     (CHOOSE with an array constant picks columns; works in Excel 2021 without CHOOSECOLS)
    fl = wb.Worksheets("flagged")
    fl.Range(fl.Cells(1, 1), fl.Cells(1, len(FLAGGED_COLUMNS))).Value = [FLAGGED_COLUMNS]
    picked = ",".join(f"tbl_products[{c}]" for c in FLAGGED_COLUMNS)
    order = ",".join(str(i) for i in range(1, len(FLAGGED_COLUMNS) + 1))
    fl.Range("A2").Formula2 = f'=FILTER(CHOOSE({{{order}}},{picked}),tbl_products[review_reason]<>"","none")'
    fl.Rows(1).Font.Bold = True
    fl.Columns.AutoFit()

    # 5b. pivot tables
    sm = wb.Worksheets("summary")
    sm.Range("A1").Value = "Products and average valid price per category"
    sm.Range("A1").Font.Bold = True
    cache = wb.PivotCaches().Create(XL_DATABASE, "tbl_products")
    pt = cache.CreatePivotTable(sm.Range("A3"), "pvt_category")
    pt.PivotFields("category").Orientation = XL_ROW_FIELD
    pt.AddDataField(pt.PivotFields("product_id"), "Products", XL_COUNT)
    avg = pt.AddDataField(pt.PivotFields("price"), "Average valid price", XL_AVERAGE)
    avg.NumberFormat = "#,##0.00"

    sm.Range("F1").Value = "Products per price status"
    sm.Range("F1").Font.Bold = True
    pt2 = wb.PivotCaches().Create(XL_DATABASE, "tbl_products").CreatePivotTable(sm.Range("F3"), "pvt_price_status")
    pt2.PivotFields("price_status").Orientation = XL_ROW_FIELD
    pt2.AddDataField(pt2.PivotFields("product_id"), "Products", XL_COUNT)
    sm.Columns.AutoFit()

    # 5c. notes
    nt = wb.Worksheets("notes")
    nt.Range(nt.Cells(1, 1), nt.Cells(len(NOTES), 1)).Value = NOTES
    for r in (1, 3, 18):
        nt.Cells(r, 1).Font.Bold = True
    nt.Cells(1, 1).Font.Size = 14

    # 6. refresh, calculate, save
    wb.RefreshAll()
    excel.CalculateUntilAsyncQueriesDone()
    excel.CalculateFull()
    nt.Activate()
    # Excel saves to a local temp file first: a synced folder (OneDrive) can lock the target while it uploads
    tmp_path = Path(tempfile.gettempdir()) / "products_clean_build.xlsx"
    tmp_path.unlink(missing_ok=True)
    wb.SaveAs(str(tmp_path), 51)
    wb.Close(False)
finally:
    excel.Quit()
    pythoncom.CoUninitialize()

for attempt in range(10):
    try:
        shutil.copyfile(tmp_path, OUT_PATH)
        break
    except PermissionError:
        time.sleep(3)  # target still locked by sync; try again
else:
    raise RuntimeError(f"could not write {OUT_PATH}")
tmp_path.unlink()
print("saved:", OUT_PATH)

saved: C:\Users\msibr\OneDrive\Upwork\customer360-data-cleaning\03_clean_data_and_report\clean_data\products_clean.xlsx


## 4. Read the workbook back and validate

The saved workbook stores the values Excel calculated, so pandas can read them and check the result independently of Excel.

In [4]:
products = pd.read_excel(OUT_PATH, sheet_name="products_clean", dtype={"price_raw": str})
flagged = pd.read_excel(OUT_PATH, sheet_name="flagged")
for col in ["category", "review_reason", "flag_type"]:
    products[col] = products[col].fillna("")
price = pd.to_numeric(products["price"], errors="coerce")

checks = {
    "500 products, same IDs as raw": len(products) == len(raw) and set(products["product_id"]) == set(raw["product_id"]),
    "product_id unique": products["product_id"].is_unique,
    "names: letters and single spaces only": products["product_name"].str.fullmatch(r"[A-Za-z]+( [A-Za-z]+)*").all(),
    "names in proper case": products["product_name"].eq(products["product_name"].str.title()).all(),
    "no 'aa' typo left": ~products["product_name"].str.lower().str.contains("aa").any(),
    "category is standard or empty": products["category"].isin(STANDARD + [""]).all(),
    "no unmapped category": products["category"].ne("UNMAPPED").all(),
    "every clean price is positive": (price.dropna() > 0).all(),
    "clean price present exactly when status is ok": price.notna().eq(products["price_status"].eq("ok")).all(),
    "flagged sheet = rows with a review reason": len(flagged) == products["review_reason"].ne("").sum(),
}
validation = pd.DataFrame({"check": checks.keys(), "passed": checks.values()})
assert validation["passed"].all(), validation
validation

,check,passed
0,"500 products, same IDs as raw",True
1,product_id unique,True
2,names: letters and single spaces only,True
3,names in proper case,True
4,no 'aa' typo left,True
5,category is standard or empty,True
6,no unmapped category,True
7,every clean price is positive,True
8,clean price present exactly when status is ok,True
9,flagged sheet = rows with a review reason,True


In [5]:
print(products["price_status"].value_counts().to_string())
print()
print(products["category"].replace("", "(missing)").value_counts().to_string())
print(f"\nflagged products: {len(flagged)}")
products.head(10)

price_status
ok                               399
invalid: repeated placeholder     30
invalid: negative                 26
invalid: not a number             26
invalid: zero                     19

category
clothing       69
kitchen        68
beauty         60
home           59
electronics    57
toys           53
automotive     52
sports         52
(missing)      30

flagged products: 112


,product_id,product_name,category_key,price_raw,price_number,category,price_status,price,review_reason,flag_type
0,PROD-0001,Open Beautiful,clo,32309.95,32309.95,clothing,ok,32309.95,,
1,PROD-0002,Race Third,automotive,47346.82,47346.82,automotive,ok,47346.82,,
2,PROD-0003,Mean Your,electronics,-100,-100.00,electronics,invalid: negative,NaN,price invalid: negative,invalid or missing value
3,PROD-0004,Consumer Newspaper,toys,28506.95,28506.95,toys,ok,28506.95,,
4,PROD-0005,A Hot,clothing,19165.58,19165.58,clothing,ok,19165.58,,
5,PROD-0006,How Over,clothing,2215.4,2215.40,clothing,ok,2215.40,,
6,PROD-0007,Each Attorney,home,49101.1,49101.10,home,ok,49101.10,,
7,PROD-0008,Idea Institution,clothing,20599.53,20599.53,clothing,ok,20599.53,,
8,PROD-0009,Billion Animal,electronics,27710.51,27710.51,electronics,ok,27710.51,,
9,PROD-0010,By Speak,clothing,0,0.00,clothing,invalid: zero,NaN,price invalid: zero,invalid or missing value


In [6]:
before_after = pd.DataFrame({
    "raw_name": raw["product_name"], "clean_name": products["product_name"],
    "raw_category": raw["category"], "clean_category": products["category"],
    "raw_price": raw["price"], "clean_price": products["price"],
})
changed = before_after[(before_after["raw_name"] != before_after["clean_name"])
                       | (before_after["raw_category"] != before_after["clean_category"])]
changed.head(15)

,raw_name,clean_name,raw_category,clean_category,raw_price,clean_price
0,Open Beaautiful,Open Beautiful,clo,clothing,32309.95,32309.95
2,MEAN YOUR,Mean Your,3l3ctronics,electronics,-100,NaN
9,By Speaak,By Speak,clothing_,clothing,0,NaN
11,Bit Its,Bit Its,electronics,electronics,35426.08,35426.08
15,StartWear,Start Wear,hom3,home,"1,200",NaN
17,current agreement,Current Agreement,CLOTHING,clothing,0,NaN
18,Section Drive,Section Drive,sports,sports,NaN,NaN
19,StockYard,Stock Yard,,,"1,200",NaN
22,Magazine Health6,Magazine Health,SPORTS,sports,0,NaN
23,Nature While,Nature While,aut,automotive,NaN,NaN


## 5. Fix log

In [7]:
CSV_DIR = CLEAN_DIR / "csv"
CSV_DIR.mkdir(exist_ok=True)
products[["product_id", "product_name", "category", "price", "price_status", "flag_type", "review_reason"]].to_csv(
    CSV_DIR / "products_clean.csv", index=False, encoding="utf-8")
print("saved CSV: products_clean.csv")

raw_name = raw["product_name"].str.strip()
clean = products["product_name"]
counts = {
    ("product_name", "leading/trailing spaces"): (raw["product_name"].ne(raw_name).sum(), 0, "Trimmed (Power Query Text.Trim)"),
    ("product_name", "noise characters"): (raw_name.str.contains(r"[^A-Za-z ]").sum(), 0, "Removed (Power Query Text.Select)"),
    ("product_name", "missing space between words"): (raw_name.str.contains(r"[a-z][A-Z]").sum(), 0,
                                                     "Split at lower-to-upper transition (Power Query)"),
    ("product_name", "doubled-letter typo 'aa'"): (raw_name.str.lower().str.contains("aa").sum(), 0, "Replaced 'aa' with 'a' (Power Query)"),
    ("product_name", "name written in all caps"): (raw_name.str.isupper().sum(), 0, "Proper case (Power Query Text.Proper)"),
    ("product_name", "name written in lower case"): (raw_name.str.islower().sum(), 0, "Proper case (Power Query Text.Proper)"),
    ("category", "leading/trailing spaces"): (raw["category"].ne(raw_cat).sum(), 0, "Trimmed (Power Query)"),
    ("category", "non-standard category value"): (
        (raw_cat.ne(products["category"]) & raw_cat.ne("")).sum(), 0,
        "Mapped to standard category (category_key in Power Query + XLOOKUP on tbl_category_map)"),
    ("category", "blank value"): (0, products["category"].eq("").sum(), "Kept empty; flagged 'category missing'"),
    ("price", "leading/trailing spaces"): (raw["price"].ne(raw["price"].str.strip()).sum(), 0, "Trimmed (Power Query)"),
    ("price", "non-numeric price"): (0, products["price_status"].eq("invalid: repeated placeholder").sum(),
                                     "Price set empty; flagged (price_status formula)"),
    ("price", "placeholder text"): (0, products["price_status"].eq("invalid: not a number").sum(),
                                    "Price set empty; flagged (price_status formula)"),
    ("price", "negative price"): (0, products["price_status"].eq("invalid: negative").sum(), "Price set empty; flagged"),
    ("price", "zero price"): (0, products["price_status"].eq("invalid: zero").sum(), "Price set empty; flagged"),
}

fix_rows = []
for (column, start), (fixed, flagged_n, action) in counts.items():
    hit = m3_issues[(m3_issues["column"] == column) & m3_issues["issue"].str.startswith(start)]
    assert len(hit) == 1, (column, start)
    row = hit.iloc[0]
    fix_rows.append({"issue_id": row["issue_id"], "column": column, "issue": row["issue"],
                     "rows_in_m1": int(row["rows_affected"]), "rows_fixed": int(fixed),
                     "rows_flagged": int(flagged_n), "action_taken": action})
fix_log = pd.DataFrame(fix_rows)
assert set(fix_log["issue_id"]) == set(m3_issues["issue_id"]), "every M3 issue must be handled"

with pd.ExcelWriter(OUT_DIR / "m3_fix_log.xlsx", engine="xlsxwriter") as writer:
    for name, data in {"fix_log": fix_log, "validation": validation, "category_map": category_map}.items():
        data.to_excel(writer, sheet_name=name, index=False)
        ws = writer.sheets[name]
        ws.freeze_panes(1, 0)
        for i, col in enumerate(data.columns):
            ws.set_column(i, i, min(max(data[col].astype(str).str.len().max(), len(col)) + 2, 70))
fix_log

saved CSV: products_clean.csv


,issue_id,column,issue,rows_in_m1,rows_fixed,rows_flagged,action_taken
0,ISS-022,product_name,leading/trailing spaces,37,37,0,Trimmed (Power Query Text.Trim)
1,ISS-025,product_name,noise characters (digits or symbols) in name,30,30,0,Removed (Power Query Text.Select)
2,ISS-029,product_name,missing space between words (e.g. 'UsuallyHear'),23,23,0,Split at lower-to-upper transition (Power Query)
3,ISS-034,product_name,doubled-letter typo 'aa' (e.g. 'Beaautiful'),11,11,0,Replaced 'aa' with 'a' (Power Query)
4,ISS-031,product_name,name written in all caps,18,18,0,Proper case (Power Query Text.Proper)
5,ISS-032,product_name,name written in lower case,17,17,0,Proper case (Power Query Text.Proper)
6,ISS-033,category,leading/trailing spaces,12,12,0,Trimmed (Power Query)
7,ISS-021,category,non-standard category value (38 variants: case...,108,96,0,Mapped to standard category (category_key in P...
8,ISS-024,category,blank value,30,0,30,Kept empty; flagged 'category missing'
9,ISS-023,price,leading/trailing spaces,35,35,0,Trimmed (Power Query)
